# Benchmark de `fit()` no LightGBM e no Bankai

Abra o notebook com o kernel da `.venv` e execute as células em ordem. Antes de medir, rode
`maturin develop -r` na raiz do repositório para instalar o Bankai em release. O kernel precisa
de `ipykernel`, `lightgbm==4.7.0`, NumPy e scikit-learn. Inicie o notebook na raiz
do repositório para importar o gerador do benchmark.

O benchmark usa o caso denso sintético de `benches/bench_train_interleaved.py`.
O cronômetro cobre a chamada pública `LGBMRegressor.fit()`, incluindo a preparação interna
dos dados. A geração dos dados e a criação do estimador ficam fora da medição.
Reinicie o kernel para aplicar a política OpenMP antes de importar as bibliotecas.

In [ ]:
import os
import subprocess
from statistics import median
from time import perf_counter

os.environ["OMP_WAIT_POLICY"] = "PASSIVE"

import lightgbm
import numpy as np
from IPython.display import Markdown, display

import bankai_lightgbm
from benches.bench_train_interleaved import CASES

if lightgbm.__version__ != "4.7.0":
    raise RuntimeError(f"Instale lightgbm==4.7.0. Versão atual: {lightgbm.__version__}")

seed = 42
rounds = 7
params = dict(n_estimators=30, n_jobs=1, num_leaves=31, random_state=seed, verbosity=-1)
X, y = CASES["dense"](np.random.default_rng(seed))
X, y = X[:10_000].copy(), y[:10_000].copy()

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"Build esperado: release | checkout: {commit}")
print(
    f"Upstream: {lightgbm.__version__} | num_threads: {params['n_jobs']} | "
    f"OMP_WAIT_POLICY: {os.environ['OMP_WAIT_POLICY']}"
)
print(f"Dados: {X.shape} | árvores solicitadas: {params['n_estimators']} | rodadas: {rounds}")

In [ ]:
libraries = {"LightGBM upstream": lightgbm, "Bankai": bankai_lightgbm}

def fit_ms(library):
    model = library.LGBMRegressor(**params)
    start = perf_counter()
    model.fit(X, y)
    return (perf_counter() - start) * 1_000

for library in libraries.values():
    fit_ms(library)

samples = {name: [] for name in libraries}
for _ in range(rounds):
    for name, library in libraries.items():
        samples[name].append(fit_ms(library))

medians = {name: median(times) for name, times in samples.items()}
rows = ["| Biblioteca | Mediana por fit | Mínimo | Máximo |", "|---|---:|---:|---:|"]
for name, times in samples.items():
    rows.append(f"| {name} | {medians[name]:.2f} ms | {min(times):.2f} ms | {max(times):.2f} ms |")
ratio = medians["Bankai"] / medians["LightGBM upstream"]
rows.extend(["", f"**Tempo Bankai / upstream:** {ratio:.2f}×"])
display(Markdown("\n".join(rows)))